# Diffusion Models for Images — Part 1: DDPM from Scratch

**Based on**: *Denoising Diffusion Probabilistic Models* (Ho et al., NeurIPS 2020),
*Deep Unsupervised Learning using Nonequilibrium Thermodynamics* (Sohl-Dickstein et al., ICML 2015),
*Generative Modeling by Estimating Gradients of the Data Distribution* (Song & Ermon, NeurIPS 2019)

---

Generative models learn to sample from a data distribution $q(x_0)$. Diffusion models achieve this
by defining a **forward process** that gradually destroys structure by adding Gaussian noise over $T$
steps, then learning a **reverse process** that removes noise step by step to generate new data.

Unlike GANs (adversarial training instability) or VAEs (posterior collapse, blurry samples), diffusion
models optimise a simple MSE loss derived from a variational bound — yet produce state-of-the-art
sample quality. The key insight of Ho et al. (2020) is that predicting the *noise* $\epsilon$ added
at each step, rather than the clean data or the denoised mean, yields a particularly simple and
effective training objective.

This notebook builds a complete DDPM pipeline from scratch on a tiny procedural image dataset,
with no external dependencies beyond torch, numpy, and matplotlib. The examples use 16×16 images
so everything runs in seconds on CPU.

This notebook explains, with runnable code:

1. The forward noising process $q(x_t \mid x_{t-1})$ and its closed-form marginal $q(x_t \mid x_0)$
2. The linear variance schedule and its effect on signal-to-noise ratio
3. The reverse denoising process $p_\theta(x_{t-1} \mid x_t)$ and the ELBO derivation
4. The simplified training loss (noise prediction)
5. A minimal UNet architecture with sinusoidal time embeddings
6. The training loop, sampling algorithm, and generated image quality

In [ ]:
import math
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim

torch.manual_seed(0)
np.random.seed(0)

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

COLORS = ["steelblue", "darkorange", "seagreen", "crimson", "mediumpurple"]

| Symbol | Meaning |
|--------|---------|
| $x_0$ | Clean data sample from $q(x_0)$ |
| $x_t$ | Noisy sample at timestep $t$ |
| $T$ | Total number of diffusion steps |
| $\beta_t$ | Noise variance added at step $t$ |
| $\alpha_t = 1 - \beta_t$ | Signal retention at step $t$ |
| $\bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s$ | Cumulative signal retention |
| $\epsilon \sim \mathcal{N}(0, \mathbf{I})$ | Standard Gaussian noise |
| $\epsilon_\theta$ | Neural network noise predictor |

---
## 1. Synthetic Image Dataset

We generate a collection of 16×16 single-channel images containing simple geometric shapes:
horizontal bars, vertical bars, crosses, and filled squares. Each pattern is created procedurally
with randomised position and thickness parameters.

This avoids any dependency on external datasets (MNIST, CIFAR) while providing enough visual
variety for the diffusion model to learn meaningful structure. The images are normalised to
$[-1, 1]$ — a standard choice for diffusion models since the forward process adds zero-mean
Gaussian noise.

In [ ]:
IMAGE_SIZE = 16


def make_horizontal_bar(image_size: int, row: int, thickness: int) -> np.ndarray:
    """Create a 2D image with a horizontal white bar on a black background."""
    img = np.zeros((image_size, image_size), dtype=np.float32)
    row_start = max(0, row - thickness // 2)
    row_end = min(image_size, row + thickness // 2 + 1)
    img[row_start:row_end, :] = 1.0
    return img


def make_vertical_bar(image_size: int, col: int, thickness: int) -> np.ndarray:
    """Create a 2D image with a vertical white bar on a black background."""
    img = np.zeros((image_size, image_size), dtype=np.float32)
    col_start = max(0, col - thickness // 2)
    col_end = min(image_size, col + thickness // 2 + 1)
    img[:, col_start:col_end] = 1.0
    return img


def make_cross(
    image_size: int, center_row: int, center_col: int, thickness: int
) -> np.ndarray:
    """Create a 2D image with a cross (overlapping horizontal and vertical bars)."""
    img_h = make_horizontal_bar(image_size, center_row, thickness)
    img_v = make_vertical_bar(image_size, center_col, thickness)
    return np.clip(img_h + img_v, 0.0, 1.0)


def make_square(image_size: int, top: int, left: int, size: int) -> np.ndarray:
    """Create a 2D image with a filled white square."""
    img = np.zeros((image_size, image_size), dtype=np.float32)
    bottom = min(image_size, top + size)
    right = min(image_size, left + size)
    img[top:bottom, left:right] = 1.0
    return img


def generate_synthetic_images(n_samples: int, image_size: int) -> torch.Tensor:
    """Generate a dataset of procedural geometric pattern images.

    Each image contains one of four pattern types: horizontal bar, vertical bar,
    cross, or filled square, with randomised position and thickness.

    Args:
        n_samples: Total number of images to generate.
        image_size: Height and width of each square image.

    Returns:
        Tensor of shape (n_samples, 1, image_size, image_size) in [-1, 1].
    """
    images = np.zeros((n_samples, image_size, image_size), dtype=np.float32)
    samples_per_pattern = n_samples // 4

    for idx in range(n_samples):
        pattern_type = idx // samples_per_pattern
        if pattern_type >= 4:
            pattern_type = np.random.randint(0, 4)

        thickness = np.random.randint(2, 5)

        if pattern_type == 0:
            row = np.random.randint(2, image_size - 2)
            images[idx] = make_horizontal_bar(image_size, row, thickness)
        elif pattern_type == 1:
            col = np.random.randint(2, image_size - 2)
            images[idx] = make_vertical_bar(image_size, col, thickness)
        elif pattern_type == 2:
            center_row = np.random.randint(3, image_size - 3)
            center_col = np.random.randint(3, image_size - 3)
            images[idx] = make_cross(image_size, center_row, center_col, thickness)
        else:
            size = np.random.randint(4, image_size // 2 + 2)
            top = np.random.randint(0, image_size - size)
            left = np.random.randint(0, image_size - size)
            images[idx] = make_square(image_size, top, left, size)

    # Normalise from [0, 1] to [-1, 1]
    images = images * 2.0 - 1.0
    images_tensor = torch.from_numpy(images).unsqueeze(1)
    return images_tensor


dataset = generate_synthetic_images(n_samples=800, image_size=IMAGE_SIZE)
print(f"Dataset shape: {dataset.shape}")
print(f"Value range  : [{dataset.min():.1f}, {dataset.max():.1f}]")

fig, axes = plt.subplots(4, 8, figsize=(10, 5.5))
for row_idx in range(4):
    for col_idx in range(8):
        sample_idx = row_idx * 200 + col_idx * 25
        axes[row_idx, col_idx].imshow(
            dataset[sample_idx, 0].numpy(), cmap="gray", vmin=-1, vmax=1
        )
        axes[row_idx, col_idx].axis("off")

row_labels = ["H-bar", "V-bar", "Cross", "Square"]
for row_idx, label in enumerate(row_labels):
    axes[row_idx, 0].set_ylabel(
        label, fontsize=10, rotation=0, labelpad=40, va="center"
    )

fig.suptitle("Synthetic 16×16 Image Dataset (4 pattern types)", y=1.01)
plt.tight_layout()
plt.show()

---
## 2. The Forward Process

The forward (noising) process gradually adds Gaussian noise to the data over $T$ timesteps.
At each step, the noisy sample $x_t$ is obtained from $x_{t-1}$ via:

$$q(x_t \mid x_{t-1}) = \mathcal{N}(x_t;\; \sqrt{1 - \beta_t}\, x_{t-1},\; \beta_t \mathbf{I})$$

The signal is scaled down by $\sqrt{1-\beta_t}$ while noise of variance $\beta_t$ is injected.
Iterating this chain would require $T$ sequential sampling steps, but a key property of Gaussians
gives us the **closed-form marginal** — we can jump directly from $x_0$ to any $x_t$:

$$q(x_t \mid x_0) = \mathcal{N}(x_t;\; \sqrt{\bar{\alpha}_t}\, x_0,\; (1 - \bar{\alpha}_t) \mathbf{I})$$

where $\bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s$ and $\alpha_s = 1 - \beta_s$.

Using the reparameterization trick:

$$x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1 - \bar{\alpha}_t}\, \epsilon, \quad \epsilon \sim \mathcal{N}(0, \mathbf{I})$$

The **linear beta schedule** sets $\beta_t$ to be linearly spaced from $\beta_1 = 10^{-4}$ to
$\beta_T = 0.02$. Early timesteps add very little noise (preserving fine structure), while later
timesteps destroy structure more aggressively. By $t = T$ the data is nearly indistinguishable
from isotropic Gaussian noise.

In [ ]:
NUM_TIMESTEPS = 300
BETA_START = 1e-4
BETA_END = 0.02


def linear_beta_schedule(
    num_timesteps: int, beta_start: float, beta_end: float
) -> torch.Tensor:
    """Create a linear variance schedule from beta_start to beta_end.

    Args:
        num_timesteps: Total number of diffusion steps T.
        beta_start: Noise variance at t=1.
        beta_end: Noise variance at t=T.

    Returns:
        Tensor of shape (num_timesteps,) with beta values.
    """
    return torch.linspace(beta_start, beta_end, num_timesteps)


def compute_alpha_bars(betas: torch.Tensor) -> torch.Tensor:
    """Compute cumulative product of (1 - beta) values.

    Args:
        betas: Tensor of shape (T,) with noise variances.

    Returns:
        Tensor of shape (T,) with alpha_bar values.
    """
    alphas = 1.0 - betas
    return torch.cumprod(alphas, dim=0)


def forward_diffusion_sample(
    x_0: torch.Tensor, timestep: torch.Tensor, alpha_bars: torch.Tensor
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Sample x_t from q(x_t | x_0) using the reparameterization trick.

    Args:
        x_0: Clean images of shape (B, C, H, W).
        timestep: Integer timestep indices of shape (B,).
        alpha_bars: Precomputed cumulative alpha products of shape (T,).

    Returns:
        Tuple of (x_t, noise) where both have shape (B, C, H, W).
    """
    alpha_bar_t = alpha_bars[timestep].view(-1, 1, 1, 1)
    noise = torch.randn_like(x_0)
    x_t = torch.sqrt(alpha_bar_t) * x_0 + torch.sqrt(1.0 - alpha_bar_t) * noise
    return x_t, noise


betas = linear_beta_schedule(NUM_TIMESTEPS, BETA_START, BETA_END)
alpha_bars = compute_alpha_bars(betas)

print(f"betas shape     : {betas.shape}")
print(f"alpha_bars shape: {alpha_bars.shape}")
print(f"\nalpha_bar at selected timesteps:")
for t_val in [0, 50, 100, 150, 200, 250, 299]:
    print(
        f"  t={t_val:>3d}  alpha_bar={alpha_bars[t_val]:.4f}  SNR={alpha_bars[t_val] / (1 - alpha_bars[t_val]):.4f}"
    )

In [ ]:
# Visualise progressive noising of a single image
sample_image = dataset[0:1]  # (1, 1, 16, 16)
display_timesteps = [0, 50, 100, 150, 200, 250, 299]

fig, axes = plt.subplots(2, 1, figsize=(10, 6), gridspec_kw={"height_ratios": [1, 1.2]})

# Top row: noised images at selected timesteps
ax_images = axes[0]
ax_images.axis("off")
inner_grid = fig.add_gridspec(2, len(display_timesteps), hspace=0.4)

# Replace top axis with sub-axes for images
ax_images.remove()
img_axes = []
for col_idx, t_val in enumerate(display_timesteps):
    ax = fig.add_subplot(inner_grid[0, col_idx])
    t_tensor = torch.tensor([t_val])
    x_t, _ = forward_diffusion_sample(sample_image, t_tensor, alpha_bars)
    ax.imshow(x_t[0, 0].numpy(), cmap="gray", vmin=-2, vmax=2)
    ax.set_title(f"t={t_val}", fontsize=9)
    ax.axis("off")
    img_axes.append(ax)

# Bottom row: alpha_bar decay curve
ax_curve = fig.add_subplot(inner_grid[1, :])
timestep_range = np.arange(NUM_TIMESTEPS)
ax_curve.plot(
    timestep_range,
    alpha_bars.numpy(),
    color=COLORS[0],
    linewidth=2,
    label=r"$\bar{\alpha}_t$",
)
ax_curve.plot(
    timestep_range,
    (1.0 - alpha_bars).numpy(),
    color=COLORS[1],
    linewidth=2,
    linestyle="--",
    label=r"$1 - \bar{\alpha}_t$ (noise level)",
)
for t_val in display_timesteps:
    ax_curve.axvline(t_val, color="gray", alpha=0.3, linewidth=0.8)
ax_curve.set_xlabel("Timestep $t$")
ax_curve.set_ylabel("Value")
ax_curve.set_title("Signal-to-Noise Ratio Decay")
ax_curve.legend(loc="center right")

plt.tight_layout()
plt.show()

---
## 3. The Reverse Process and ELBO

Sampling new data requires reversing the forward process — denoising from $x_T \sim \mathcal{N}(0, \mathbf{I})$
back to $x_0$. The true reverse distribution $q(x_{t-1} \mid x_t)$ is intractable, so we learn an
approximation:

$$p_\theta(x_{t-1} \mid x_t) = \mathcal{N}(x_{t-1};\; \mu_\theta(x_t, t),\; \sigma_t^2 \mathbf{I})$$

When $x_0$ is known, the **tractable posterior** has a closed form:

$$q(x_{t-1} \mid x_t, x_0) = \mathcal{N}\!\left(x_{t-1};\; \tilde{\mu}_t(x_t, x_0),\; \tilde{\beta}_t \mathbf{I}\right)$$

with posterior mean:

$$\tilde{\mu}_t = \frac{\sqrt{\bar{\alpha}_{t-1}}\,\beta_t}{1 - \bar{\alpha}_t} x_0 + \frac{\sqrt{\alpha_t}(1 - \bar{\alpha}_{t-1})}{1 - \bar{\alpha}_t} x_t$$

and posterior variance $\tilde{\beta}_t = \frac{1 - \bar{\alpha}_{t-1}}{1 - \bar{\alpha}_t} \beta_t$.

The ELBO decomposes into per-timestep KL terms between $q(x_{t-1}|x_t,x_0)$ and $p_\theta(x_{t-1}|x_t)$.
Since both are Gaussian with the same variance, matching them reduces to matching means.
Substituting the reparameterization $x_0 = \frac{1}{\sqrt{\bar{\alpha}_t}}(x_t - \sqrt{1-\bar{\alpha}_t}\,\epsilon)$
and rearranging, Ho et al. show that the optimal $\mu_\theta$ depends only on predicting $\epsilon$.

The **simplified training loss** (Ho et al. Eq. 14) drops the timestep-dependent weighting:

$$L_\text{simple} = \mathbb{E}_{t, x_0, \epsilon}\!\left[\|\epsilon - \epsilon_\theta(x_t, t)\|^2\right]$$

**Why epsilon-prediction is preferred**: predicting $\epsilon$ rather than $x_0$ or $\tilde{\mu}_t$
directly provides a more uniform loss scale across timesteps. At large $t$, $x_t$ is mostly noise
so predicting $x_0$ from it has very high variance; predicting the noise that was added is
numerically better-conditioned.

---
## 4. Minimal UNet Architecture

The noise predictor $\epsilon_\theta(x_t, t)$ takes a noisy image and a timestep and outputs
a noise estimate with the same spatial dimensions. We use a minimal UNet:

- **Sinusoidal time embedding**: Reuses the positional encoding from transformers to convert the
  scalar timestep $t$ into a dense vector. This is projected through two linear layers with SiLU
  activation.

- **Encoder**: 2 down-blocks, each containing Conv2d (stride 2 for downsampling) followed by
  GroupNorm and SiLU. The projected time embedding is added after the first convolution.
  Spatial dimensions halve at each block: 16→8→4.

- **Bottleneck**: A Conv block at 4×4 spatial resolution.

- **Decoder**: 2 up-blocks using ConvTranspose2d (stride 2) for upsampling. Skip connections
  from the encoder are concatenated along the channel dimension before the transpose convolution.
  Spatial dimensions double: 4→8→16.

- **Output**: 1×1 Conv to project back to 1 channel.

Channel progression: 1 → 32 → 64 → 64 → 32 → 1

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    """Sinusoidal positional encoding for diffusion timesteps."""

    def __init__(self, embedding_dim: int) -> None:
        super().__init__()
        self.embedding_dim = embedding_dim

    def forward(self, timesteps: torch.Tensor) -> torch.Tensor:
        """Convert integer timesteps to sinusoidal embeddings.

        Args:
            timesteps: Integer tensor of shape (B,).

        Returns:
            Embeddings of shape (B, embedding_dim).
        """
        half_dim = self.embedding_dim // 2
        freq_scale = math.log(10000.0) / (half_dim - 1)
        frequencies = torch.exp(
            torch.arange(half_dim, device=timesteps.device, dtype=torch.float32)
            * -freq_scale
        )
        angles = timesteps.float().unsqueeze(1) * frequencies.unsqueeze(0)
        return torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)


class DownBlock(nn.Module):
    """Conv2d downsampling block with time conditioning."""

    def __init__(self, in_channels: int, out_channels: int, time_dim: int) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(
            in_channels, out_channels, kernel_size=3, stride=2, padding=1
        )
        self.norm1 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.act1 = nn.SiLU()

        self.time_proj = nn.Linear(time_dim, out_channels)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.norm2 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.act2 = nn.SiLU()

    def forward(self, x: torch.Tensor, t_emb: torch.Tensor) -> torch.Tensor:
        """Downsample with time conditioning.

        Args:
            x: Input tensor of shape (B, C_in, H, W).
            t_emb: Time embedding of shape (B, time_dim).

        Returns:
            Output tensor of shape (B, C_out, H//2, W//2).
        """
        hidden = self.act1(self.norm1(self.conv1(x)))
        time_signal = self.time_proj(t_emb).unsqueeze(-1).unsqueeze(-1)
        hidden = hidden + time_signal
        hidden = self.act2(self.norm2(self.conv2(hidden)))
        return hidden


class UpBlock(nn.Module):
    """ConvTranspose2d upsampling block with skip connection and time conditioning."""

    def __init__(
        self, in_channels: int, skip_channels: int, out_channels: int, time_dim: int
    ) -> None:
        super().__init__()
        self.up_conv = nn.ConvTranspose2d(
            in_channels + skip_channels,
            out_channels,
            kernel_size=4,
            stride=2,
            padding=1,
        )
        self.norm1 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.act1 = nn.SiLU()

        self.time_proj = nn.Linear(time_dim, out_channels)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.norm2 = nn.GroupNorm(num_groups=8, num_channels=out_channels)
        self.act2 = nn.SiLU()

    def forward(
        self, x: torch.Tensor, skip: torch.Tensor, t_emb: torch.Tensor
    ) -> torch.Tensor:
        """Upsample with skip connection and time conditioning.

        Args:
            x: Input tensor of shape (B, C_in, H, W).
            skip: Skip connection from encoder of shape (B, C_skip, 2*H, 2*W).
            t_emb: Time embedding of shape (B, time_dim).

        Returns:
            Output tensor of shape (B, C_out, 2*H, 2*W).
        """
        combined = torch.cat([x, skip], dim=1)
        hidden = self.act1(self.norm1(self.up_conv(combined)))
        time_signal = self.time_proj(t_emb).unsqueeze(-1).unsqueeze(-1)
        hidden = hidden + time_signal
        hidden = self.act2(self.norm2(self.conv2(hidden)))
        return hidden


class SimpleUNet(nn.Module):
    """Minimal UNet for 16x16 single-channel images with time conditioning."""

    def __init__(
        self, image_channels: int = 1, base_channels: int = 32, time_dim: int = 64
    ) -> None:
        super().__init__()

        self.time_embed = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim),
        )

        self.init_conv = nn.Conv2d(
            image_channels, base_channels, kernel_size=3, padding=1
        )

        # Encoder: 16x16 -> 8x8 -> 4x4
        self.down1 = DownBlock(base_channels, base_channels, time_dim)  # 32->32, 16->8
        self.down2 = DownBlock(
            base_channels, base_channels * 2, time_dim
        )  # 32->64, 8->4

        # Bottleneck at 4x4
        self.bottleneck_conv = nn.Conv2d(
            base_channels * 2, base_channels * 2, kernel_size=3, padding=1
        )
        self.bottleneck_norm = nn.GroupNorm(
            num_groups=8, num_channels=base_channels * 2
        )
        self.bottleneck_act = nn.SiLU()

        # Decoder: 4x4 -> 8x8 -> 16x16
        # up1 takes bottleneck (64ch) + skip from down2 (64ch)
        self.up1 = UpBlock(
            base_channels * 2, base_channels * 2, base_channels, time_dim
        )
        # up2 takes up1 output (32ch) + skip from down1 (32ch)
        self.up2 = UpBlock(base_channels, base_channels, base_channels, time_dim)

        self.final_conv = nn.Conv2d(base_channels, image_channels, kernel_size=1)

    def forward(self, x: torch.Tensor, timesteps: torch.Tensor) -> torch.Tensor:
        """Predict noise from noisy image and timestep.

        Args:
            x: Noisy images of shape (B, C, H, W).
            timesteps: Integer timestep indices of shape (B,).

        Returns:
            Predicted noise of shape (B, C, H, W).
        """
        t_emb = self.time_embed(timesteps)

        h0 = self.init_conv(x)  # (B, 32, 16, 16)
        h1 = self.down1(h0, t_emb)  # (B, 32, 8, 8)
        h2 = self.down2(h1, t_emb)  # (B, 64, 4, 4)

        h_mid = self.bottleneck_act(
            self.bottleneck_norm(self.bottleneck_conv(h2))
        )  # (B, 64, 4, 4)

        h3 = self.up1(h_mid, h2, t_emb)  # (B, 32, 8, 8)
        h4 = self.up2(h3, h1, t_emb)  # (B, 32, 16, 16)

        return self.final_conv(h4)  # (B, 1, 16, 16)

In [ ]:
model = SimpleUNet()
dummy_x = torch.randn(4, 1, 16, 16)
dummy_t = torch.randint(0, NUM_TIMESTEPS, (4,))
output = model(dummy_x, dummy_t)
print(f"Input shape : {dummy_x.shape}")
print(f"Output shape: {output.shape}")
assert output.shape == dummy_x.shape


def count_parameters(module: nn.Module) -> int:
    """Count total trainable parameters in a module."""
    return sum(p.numel() for p in module.parameters())


print(f"\nTotal parameters: {count_parameters(model):,}")

---
## 5. Training

The DDPM training procedure (Algorithm 1 from Ho et al.) is remarkably simple:

1. Sample a clean image $x_0 \sim q(x_0)$ from the dataset
2. Sample a random timestep $t \sim \text{Uniform}(\{1, \ldots, T\})$
3. Sample noise $\epsilon \sim \mathcal{N}(0, \mathbf{I})$
4. Compute the noisy image $x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1 - \bar{\alpha}_t}\, \epsilon$
5. Take a gradient step on $\|\epsilon - \epsilon_\theta(x_t, t)\|^2$

There is no adversarial training, no reconstruction term, no posterior collapse — just MSE between
the true noise and the predicted noise.

In [ ]:
def train_diffusion(
    model: SimpleUNet,
    dataset: torch.Tensor,
    alpha_bars: torch.Tensor,
    num_timesteps: int,
    n_steps: int = 2000,
    batch_size: int = 64,
    lr: float = 1e-3,
) -> Dict[str, List[float]]:
    """Train DDPM noise predictor.

    Args:
        model: UNet noise prediction network.
        dataset: Training images of shape (N, 1, H, W) in [-1, 1].
        alpha_bars: Precomputed cumulative alpha products of shape (T,).
        num_timesteps: Total number of diffusion steps T.
        n_steps: Number of training iterations.
        batch_size: Minibatch size.
        lr: Learning rate.

    Returns:
        Dictionary with 'step' and 'loss' lists for tracking.
    """
    optimizer = optim.Adam(model.parameters(), lr=lr)
    n_samples = dataset.shape[0]
    history: Dict[str, List[float]] = {"step": [], "loss": []}

    model.train()
    for step_idx in range(n_steps):
        optimizer.zero_grad()

        batch_indices = torch.randint(0, n_samples, (batch_size,))
        x_0 = dataset[batch_indices]

        timestep = torch.randint(0, num_timesteps, (batch_size,))

        x_t, noise = forward_diffusion_sample(x_0, timestep, alpha_bars)

        noise_pred = model(x_t, timestep)
        loss = F.mse_loss(noise_pred, noise)

        loss.backward()
        optimizer.step()

        if step_idx % 50 == 0:
            history["step"].append(step_idx)
            history["loss"].append(loss.item())

        if step_idx % 500 == 0:
            print(f"  step {step_idx:>5d}  loss={loss.item():.4f}")

    return history


trained_model = SimpleUNet()
print("Training DDPM noise predictor...")
history = train_diffusion(
    trained_model,
    dataset,
    alpha_bars,
    NUM_TIMESTEPS,
    n_steps=2000,
    batch_size=64,
    lr=1e-3,
)
print(f"\nFinal loss: {history['loss'][-1]:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(history["step"], history["loss"], color=COLORS[0], linewidth=1.5)
ax.set_xlabel("Training Step")
ax.set_ylabel("MSE Loss (log scale)")
ax.set_title("DDPM Training Loss")
plt.tight_layout()
plt.show()

---
## 6. Sampling (Reverse Process)

Sampling (Algorithm 2 from Ho et al.) starts from pure noise $x_T \sim \mathcal{N}(0, \mathbf{I})$
and iteratively denoises:

$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}}\,\epsilon_\theta(x_t, t)\right) + \sigma_t\, z$$

where $z \sim \mathcal{N}(0, \mathbf{I})$ if $t > 1$, else $z = 0$, and $\sigma_t = \sqrt{\beta_t}$.

This requires $T$ sequential forward passes through the network — one of the main computational
drawbacks of diffusion models compared to single-pass generators like GANs.

In [ ]:
@torch.no_grad()
def sample(
    model: SimpleUNet,
    n_samples: int,
    image_shape: Tuple[int, int, int],
    betas: torch.Tensor,
    alpha_bars: torch.Tensor,
) -> torch.Tensor:
    """Generate samples via the reverse diffusion process.

    Args:
        model: Trained noise prediction network.
        n_samples: Number of images to generate.
        image_shape: (C, H, W) tuple.
        betas: Noise schedule of shape (T,).
        alpha_bars: Cumulative alpha products of shape (T,).

    Returns:
        Generated images of shape (n_samples, C, H, W).
    """
    model.eval()
    num_timesteps = len(betas)
    alphas = 1.0 - betas

    x_t = torch.randn(n_samples, *image_shape)

    for t_idx in reversed(range(num_timesteps)):
        t_batch = torch.full((n_samples,), t_idx, dtype=torch.long)

        eps_pred = model(x_t, t_batch)

        alpha_t = alphas[t_idx]
        alpha_bar_t = alpha_bars[t_idx]
        beta_t = betas[t_idx]

        mean = (1.0 / torch.sqrt(alpha_t)) * (
            x_t - (beta_t / torch.sqrt(1.0 - alpha_bar_t)) * eps_pred
        )

        if t_idx > 0:
            noise = torch.randn_like(x_t)
            sigma_t = torch.sqrt(beta_t)
            x_t = mean + sigma_t * noise
        else:
            x_t = mean

    return x_t


print("Generating 32 samples...")
generated = sample(trained_model, 32, (1, IMAGE_SIZE, IMAGE_SIZE), betas, alpha_bars)
print(f"Generated shape: {generated.shape}")

fig, all_axes = plt.subplots(4, 16, figsize=(16, 5))

# Left 8 columns: generated samples
for row_idx in range(4):
    for col_idx in range(8):
        sample_idx = row_idx * 8 + col_idx
        all_axes[row_idx, col_idx].imshow(
            generated[sample_idx, 0].numpy(), cmap="gray", vmin=-1, vmax=1
        )
        all_axes[row_idx, col_idx].axis("off")

# Right 8 columns: real samples
for row_idx in range(4):
    for col_idx in range(8):
        real_idx = row_idx * 200 + col_idx * 20
        all_axes[row_idx, col_idx + 8].imshow(
            dataset[real_idx, 0].numpy(), cmap="gray", vmin=-1, vmax=1
        )
        all_axes[row_idx, col_idx + 8].axis("off")

# Column labels
all_axes[0, 3].set_title("Generated", fontsize=11, pad=8)
all_axes[0, 11].set_title("Real", fontsize=11, pad=8)

plt.tight_layout()
plt.show()

In [ ]:
# Progressive denoising visualisation: show intermediate states for one sample
@torch.no_grad()
def sample_with_intermediates(
    model: SimpleUNet,
    image_shape: Tuple[int, int, int],
    betas: torch.Tensor,
    alpha_bars: torch.Tensor,
    save_at_timesteps: List[int],
) -> Dict[int, torch.Tensor]:
    """Generate one sample, saving intermediate states at specified timesteps.

    Args:
        model: Trained noise prediction network.
        image_shape: (C, H, W) tuple.
        betas: Noise schedule of shape (T,).
        alpha_bars: Cumulative alpha products of shape (T,).
        save_at_timesteps: Timestep values at which to save the current state.

    Returns:
        Dictionary mapping timestep to image tensor of shape (C, H, W).
    """
    model.eval()
    num_timesteps = len(betas)
    alphas = 1.0 - betas
    save_set = set(save_at_timesteps)
    intermediates: Dict[int, torch.Tensor] = {}

    x_t = torch.randn(1, *image_shape)

    for t_idx in reversed(range(num_timesteps)):
        if t_idx in save_set:
            intermediates[t_idx] = x_t[0].clone()

        t_batch = torch.full((1,), t_idx, dtype=torch.long)
        eps_pred = model(x_t, t_batch)

        alpha_t = alphas[t_idx]
        alpha_bar_t = alpha_bars[t_idx]
        beta_t = betas[t_idx]

        mean = (1.0 / torch.sqrt(alpha_t)) * (
            x_t - (beta_t / torch.sqrt(1.0 - alpha_bar_t)) * eps_pred
        )

        if t_idx > 0:
            noise = torch.randn_like(x_t)
            sigma_t = torch.sqrt(beta_t)
            x_t = mean + sigma_t * noise
        else:
            x_t = mean

    intermediates[0] = x_t[0].clone()
    return intermediates


snapshot_timesteps = [
    NUM_TIMESTEPS - 1,
    int(0.75 * NUM_TIMESTEPS),
    int(0.50 * NUM_TIMESTEPS),
    int(0.25 * NUM_TIMESTEPS),
    int(0.10 * NUM_TIMESTEPS),
]

intermediates = sample_with_intermediates(
    trained_model, (1, IMAGE_SIZE, IMAGE_SIZE), betas, alpha_bars, snapshot_timesteps
)

display_t_values = sorted(intermediates.keys(), reverse=True)
n_snapshots = len(display_t_values)

fig, axes = plt.subplots(1, n_snapshots, figsize=(2.2 * n_snapshots, 2.5))
for ax_idx, t_val in enumerate(display_t_values):
    axes[ax_idx].imshow(intermediates[t_val][0].numpy(), cmap="gray", vmin=-2, vmax=2)
    axes[ax_idx].set_title(f"t={t_val}", fontsize=10)
    axes[ax_idx].axis("off")

fig.suptitle("Progressive Denoising: From Pure Noise to Generated Image", y=1.02)
plt.tight_layout()
plt.show()

---
## 7. Exercises

### Exercise 1: Cosine Beta Schedule

Nichol & Dhariwal (ICML 2021) proposed a cosine schedule that provides more gradual noising:

$$\bar{\alpha}_t = \frac{f(t/T)}{f(0)}, \quad f(s) = \cos^2\!\left(\frac{s + 0.008}{1.008}\cdot\frac{\pi}{2}\right)$$

Implement `cosine_beta_schedule` below.

In [ ]:
def cosine_beta_schedule(num_timesteps: int, max_beta: float = 0.999) -> torch.Tensor:
    """Cosine beta schedule (Nichol & Dhariwal, 2021).

    Args:
        num_timesteps: Total number of diffusion steps T.
        max_beta: Clamp betas to this maximum value.

    Returns:
        Tensor of shape (num_timesteps,) with beta values.
    """
    # YOUR CODE HERE
    pass

In [ ]:
cosine_betas = cosine_beta_schedule(NUM_TIMESTEPS)
assert cosine_betas.shape == (NUM_TIMESTEPS,), f"Expected shape ({NUM_TIMESTEPS},)"
cosine_alpha_bars = compute_alpha_bars(cosine_betas)
assert torch.all(cosine_alpha_bars[:-1] >= cosine_alpha_bars[1:]), (
    "alpha_bars must be monotonically decreasing"
)
assert cosine_alpha_bars[0] > 0.99, (
    f"alpha_bar[0] should be near 1, got {cosine_alpha_bars[0]:.4f}"
)
assert cosine_alpha_bars[-1] < 0.05, (
    f"alpha_bar[T-1] should be near 0, got {cosine_alpha_bars[-1]:.4f}"
)
print(
    f"Cosine alpha_bar range: [{cosine_alpha_bars[-1]:.4f}, {cosine_alpha_bars[0]:.4f}]"
)
print("Exercise 1 passed!")

### Exercise 2: DDIM Sampling

Song et al. (ICLR 2021) showed that the same trained model can be sampled with a
*deterministic* process using fewer steps. The DDIM update rule for a subsequence
of timesteps $\tau_1 > \tau_2 > \cdots > \tau_S$ with $\eta = 0$ (fully deterministic) is:

$$x_{\tau_{s-1}} = \sqrt{\bar{\alpha}_{\tau_{s-1}}}\,\hat{x}_0 + \sqrt{1 - \bar{\alpha}_{\tau_{s-1}}}\,\frac{x_{\tau_s} - \sqrt{\bar{\alpha}_{\tau_s}}\,\hat{x}_0}{\sqrt{1 - \bar{\alpha}_{\tau_s}}}$$

where $\hat{x}_0 = \frac{x_{\tau_s} - \sqrt{1 - \bar{\alpha}_{\tau_s}}\,\epsilon_\theta(x_{\tau_s}, \tau_s)}{\sqrt{\bar{\alpha}_{\tau_s}}}$.

Implement `sample_ddim` using only 50 evenly spaced steps from the 300 total.

In [ ]:
@torch.no_grad()
def sample_ddim(
    model: SimpleUNet,
    n_samples: int,
    image_shape: Tuple[int, int, int],
    alpha_bars: torch.Tensor,
    num_ddim_steps: int = 50,
) -> torch.Tensor:
    """Deterministic DDIM sampling with a reduced number of steps.

    Args:
        model: Trained noise prediction network.
        n_samples: Number of images to generate.
        image_shape: (C, H, W) tuple.
        alpha_bars: Precomputed cumulative alpha products.
        num_ddim_steps: Number of denoising steps (< T).

    Returns:
        Generated images of shape (n_samples, C, H, W).
    """
    # YOUR CODE HERE
    pass

In [ ]:
ddim_samples = sample_ddim(trained_model, 8, (1, IMAGE_SIZE, IMAGE_SIZE), alpha_bars)
assert ddim_samples.shape == (8, 1, IMAGE_SIZE, IMAGE_SIZE), (
    f"Wrong shape: {ddim_samples.shape}"
)
print(f"DDIM samples shape: {ddim_samples.shape}")
print("Exercise 2 passed!")

### Exercise 3: Conditional UNet

Extend `SimpleUNet` to accept a **class label** (the pattern type: 0=horizontal bar, 1=vertical
bar, 2=cross, 3=square). The label is embedded via `nn.Embedding(num_classes, time_dim)` and
added to the time embedding before being passed to each block.

This is the simplest form of conditional generation — the model learns to generate images of
a requested class.

In [ ]:
class ConditionalUNet(nn.Module):
    """UNet with class-conditional generation via label embedding.

    The class label is embedded and added to the time embedding,
    so each block receives both timestep and class information.
    """

    def __init__(
        self,
        image_channels: int = 1,
        base_channels: int = 32,
        time_dim: int = 64,
        num_classes: int = 4,
    ) -> None:
        super().__init__()
        # YOUR CODE HERE
        pass

    def forward(
        self, x: torch.Tensor, timesteps: torch.Tensor, labels: torch.Tensor
    ) -> torch.Tensor:
        """Predict noise conditioned on timestep and class label.

        Args:
            x: Noisy images of shape (B, C, H, W).
            timesteps: Integer timestep indices of shape (B,).
            labels: Integer class labels of shape (B,).

        Returns:
            Predicted noise of shape (B, C, H, W).
        """
        # YOUR CODE HERE
        pass

In [ ]:
cond_model = ConditionalUNet()
dummy_x = torch.randn(4, 1, 16, 16)
dummy_t = torch.randint(0, NUM_TIMESTEPS, (4,))
dummy_labels = torch.tensor([0, 1, 2, 3])
cond_output = cond_model(dummy_x, dummy_t, dummy_labels)
assert cond_output.shape == dummy_x.shape, f"Wrong shape: {cond_output.shape}"
print(f"ConditionalUNet output shape: {cond_output.shape}")
print(f"ConditionalUNet parameters : {count_parameters(cond_model):,}")
print("Exercise 3 passed!")

---
## 8. Challenges and Limitations

- **Slow sampling**: Generating one image requires $T$ sequential forward passes through the
  network. With $T=1000$ (the original DDPM), this is 1000× slower than a single-pass GAN.
  DDIM (Exercise 2) and distillation methods reduce this but introduce approximation error.

- **No conditioning**: This implementation is unconditional — it cannot generate images of a
  specific class or matching a text prompt. Classifier-free guidance (Ho & Salimans, 2022)
  adds controllability by jointly training conditional and unconditional models.

- **Scale**: 16×16 is far too small for realistic images. The original DDPM uses 64×64 with
  much deeper UNets (~35M parameters). Latent diffusion (Rombach et al., CVPR 2022) moves
  the diffusion process to a lower-dimensional VAE latent space to handle higher resolutions.

- **Training stability**: With small datasets, the per-step MSE loss is noisy. Exponential
  moving average (EMA) of model weights stabilises generation quality at the cost of slightly
  slower convergence tracking.

- **Memory**: Skip connections in the UNet store activations at every spatial scale during the
  forward pass. For high-resolution images with many scales, this dominates GPU memory.

In [ ]:
# Computational cost analysis at different scales
print("Parameter counts at different image/channel configurations:")
print(f"{'Config':<35s}  {'Params':>10s}")
print("-" * 48)

configs = [
    ("16×16, 1ch, base=32 (this tutorial)", 1, 32),
    ("16×16, 1ch, base=64", 1, 64),
    ("16×16, 3ch, base=32", 3, 32),
    ("16×16, 3ch, base=64", 3, 64),
    ("16×16, 3ch, base=128", 3, 128),
]

for label, img_ch, base_ch in configs:
    test_model = SimpleUNet(image_channels=img_ch, base_channels=base_ch)
    n_params = count_parameters(test_model)
    print(f"  {label:<33s}  {n_params:>10,}")

print(f"\nSampling cost: {NUM_TIMESTEPS} forward passes per image")
print(
    f"DDIM (50 steps): {50} forward passes per image ({NUM_TIMESTEPS / 50:.0f}× speedup)"
)

---
## 9. Summary

```
Core idea:
  Destroy data by adding Gaussian noise over T steps (forward process),
  then learn to reverse it (reverse process) to generate new data.

Forward process:
  q(x_t | x_0) = N(sqrt(alpha_bar_t) * x_0, (1 - alpha_bar_t) * I)
  Closed-form marginal avoids sequential sampling during training.

Reverse process:
  p_theta(x_{t-1} | x_t) = N(mu_theta(x_t, t), sigma_t^2 * I)
  Learned mean parameterised by noise prediction epsilon_theta.

Training:
  L_simple = E[||epsilon - epsilon_theta(x_t, t)||^2]
  Sample random t, noise x_0, predict the noise. MSE loss.

Architecture:
  UNet with sinusoidal time embedding, skip connections,
  GroupNorm + SiLU activations. Output same shape as input.

Sampling:
  Start from x_T ~ N(0, I), iterate T reverse steps.
  Each step: predict noise, compute mean, add scaled noise (except t=0).

Key tradeoffs:
  + Simple MSE loss, stable training, high sample quality
  - Slow sampling (T forward passes), no conditioning by default
  - Memory-intensive UNet skip connections at high resolution

Extensions:
  - Cosine schedule: more gradual noising (Nichol & Dhariwal 2021)
  - DDIM: deterministic sampling with fewer steps (Song et al. 2021)
  - Classifier-free guidance: conditional generation (Ho & Salimans 2022)
  - Latent diffusion: diffuse in VAE latent space (Rombach et al. 2022)
```

---
## Appendix: Exercise Solutions

Expand the cells below only after attempting the exercises.

In [ ]:
# Solution 1: Cosine beta schedule


def cosine_beta_schedule_solution(
    num_timesteps: int, max_beta: float = 0.999
) -> torch.Tensor:
    """Cosine beta schedule (Nichol & Dhariwal, 2021)."""
    steps = torch.arange(num_timesteps + 1, dtype=torch.float64)

    def f_cosine(s: torch.Tensor) -> torch.Tensor:
        return torch.cos(((s / num_timesteps) + 0.008) / 1.008 * (math.pi / 2.0)) ** 2

    alpha_bars_cos = f_cosine(steps) / f_cosine(torch.tensor(0.0, dtype=torch.float64))

    # beta_t = 1 - alpha_bar_t / alpha_bar_{t-1}
    result_betas = 1.0 - (alpha_bars_cos[1:] / alpha_bars_cos[:-1])
    result_betas = torch.clamp(result_betas, max=max_beta)
    return result_betas.float()


# Verify the solution
sol1_betas = cosine_beta_schedule_solution(NUM_TIMESTEPS)
sol1_alpha_bars = compute_alpha_bars(sol1_betas)
print(
    f"Solution 1: cosine alpha_bar range [{sol1_alpha_bars[-1]:.4f}, {sol1_alpha_bars[0]:.4f}]"
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(alpha_bars.numpy(), label="Linear schedule", color=COLORS[0], linewidth=2)
ax.plot(sol1_alpha_bars.numpy(), label="Cosine schedule", color=COLORS[1], linewidth=2)
ax.set_xlabel("Timestep $t$")
ax.set_ylabel(r"$\bar{\alpha}_t$")
ax.set_title("Linear vs Cosine Schedule")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Solution 2: DDIM sampling


@torch.no_grad()
def sample_ddim_solution(
    model: SimpleUNet,
    n_samples: int,
    image_shape: Tuple[int, int, int],
    alpha_bars: torch.Tensor,
    num_ddim_steps: int = 50,
) -> torch.Tensor:
    """Deterministic DDIM sampling with a reduced number of steps."""
    model.eval()
    num_timesteps = len(alpha_bars)

    # Evenly spaced subsequence of timesteps
    step_size = num_timesteps // num_ddim_steps
    timestep_seq = list(range(0, num_timesteps, step_size))

    x_t = torch.randn(n_samples, *image_shape)

    # Iterate from largest to smallest timestep
    for seq_idx in reversed(range(len(timestep_seq))):
        t_current = timestep_seq[seq_idx]
        t_batch = torch.full((n_samples,), t_current, dtype=torch.long)

        eps_pred = model(x_t, t_batch)

        alpha_bar_t = alpha_bars[t_current]

        # Predict x_0 from current x_t and predicted noise
        x_0_pred = (x_t - torch.sqrt(1.0 - alpha_bar_t) * eps_pred) / torch.sqrt(
            alpha_bar_t
        )

        if seq_idx > 0:
            t_prev = timestep_seq[seq_idx - 1]
            alpha_bar_prev = alpha_bars[t_prev]

            # Direction pointing to x_t
            direction = torch.sqrt(1.0 - alpha_bar_prev) * (
                (x_t - torch.sqrt(alpha_bar_t) * x_0_pred)
                / torch.sqrt(1.0 - alpha_bar_t)
            )

            x_t = torch.sqrt(alpha_bar_prev) * x_0_pred + direction
        else:
            x_t = x_0_pred

    return x_t


print("Generating 8 samples with DDIM (50 steps)...")
sol2_samples = sample_ddim_solution(
    trained_model, 8, (1, IMAGE_SIZE, IMAGE_SIZE), alpha_bars
)
print(f"DDIM samples shape: {sol2_samples.shape}")

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax_idx in range(8):
    axes[ax_idx].imshow(sol2_samples[ax_idx, 0].numpy(), cmap="gray", vmin=-1, vmax=1)
    axes[ax_idx].axis("off")
fig.suptitle("DDIM Samples (50 steps instead of 300)", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Solution 3: ConditionalUNet


class ConditionalUNetSolution(nn.Module):
    """UNet with class-conditional generation via label embedding."""

    def __init__(
        self,
        image_channels: int = 1,
        base_channels: int = 32,
        time_dim: int = 64,
        num_classes: int = 4,
    ) -> None:
        super().__init__()

        self.time_embed = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim),
        )

        self.class_embed = nn.Embedding(num_classes, time_dim)

        self.init_conv = nn.Conv2d(
            image_channels, base_channels, kernel_size=3, padding=1
        )

        self.down1 = DownBlock(base_channels, base_channels, time_dim)
        self.down2 = DownBlock(base_channels, base_channels * 2, time_dim)

        self.bottleneck_conv = nn.Conv2d(
            base_channels * 2, base_channels * 2, kernel_size=3, padding=1
        )
        self.bottleneck_norm = nn.GroupNorm(
            num_groups=8, num_channels=base_channels * 2
        )
        self.bottleneck_act = nn.SiLU()

        self.up1 = UpBlock(
            base_channels * 2, base_channels * 2, base_channels, time_dim
        )
        self.up2 = UpBlock(base_channels, base_channels, base_channels, time_dim)

        self.final_conv = nn.Conv2d(base_channels, image_channels, kernel_size=1)

    def forward(
        self, x: torch.Tensor, timesteps: torch.Tensor, labels: torch.Tensor
    ) -> torch.Tensor:
        """Predict noise conditioned on timestep and class label."""
        t_emb = self.time_embed(timesteps) + self.class_embed(labels)

        h0 = self.init_conv(x)
        h1 = self.down1(h0, t_emb)
        h2 = self.down2(h1, t_emb)

        h_mid = self.bottleneck_act(self.bottleneck_norm(self.bottleneck_conv(h2)))

        h3 = self.up1(h_mid, h2, t_emb)
        h4 = self.up2(h3, h1, t_emb)

        return self.final_conv(h4)


sol3_model = ConditionalUNetSolution()
sol3_out = sol3_model(
    torch.randn(4, 1, 16, 16),
    torch.randint(0, NUM_TIMESTEPS, (4,)),
    torch.tensor([0, 1, 2, 3]),
)
print(f"ConditionalUNet output shape: {sol3_out.shape}")
print(f"ConditionalUNet parameters : {count_parameters(sol3_model):,}")

---
**Next**: [Part 2 — Tabular Diffusion](./part2-tabular-diffusion.ipynb) covers Gaussian and
multinomial diffusion for mixed continuous and categorical tabular data.